# Do Shoppers Follow the Catalogue?
### Product communities, network analysis and machine learning from Instacart baskets

TUM Heilbronn · Data Science (CITHN1007) · WS 2026/27 · Syed Wassi Ul Haque & Nicolas Ignacio Rivolta

**Main question:** Do co-purchase product communities outperform catalogue categories (departments / aisles) in capturing
real shopping behaviour, and can those communities guide assortment decisions in small-format retail?

| # | Sub-question | Lead |
|---|---|---|
| SQ1 | Louvain communities vs aisles / departments | Syed |
| SQ2 | Groups per basket: communities vs departments | Nicolas |
| SQ3 | Most central products | Syed |
| SQ4 | Mini-market of N products: which strategy serves most baskets? | Nicolas |
| SQ5 [ML] | Does community membership improve reorder prediction? | Syed |
| SQ6 [ML] | K-means shopper segments from community-purchase vectors | Nicolas |

This notebook must run **top to bottom from the raw data** in `data/` and create every figure.

## Contents
0. Setup
1. Step 0 — Data preparation (load, joins, 80/20 user split, top-N products)
2. Step 1 — Exploratory data analysis
3. Step 2 — The co-purchase network
4. SQ1 — Communities vs the catalogue
5. SQ2 — Groups per basket
6. SQ3 — Central products
7. SQ4 — The mini-market
8. SQ5 [ML] — Predicting reorders
9. SQ6 [ML] — Shopper segments
10. Robustness checks
11. Conclusions and data story

## 0. Setup
Imports, fixed random seed, relative paths, output folders. Guide: chapter "How to use this guide".

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import networkx as nx
import matplotlib.pyplot as plt

SEED = 42                      # one seed for everything (reproducibility)
# The notebook lives in notebook/, the data in the repo root: find the root either way.
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA = ROOT / "data"           # relative path only, never a personal path
FIG = ROOT / "figures"
RES = ROOT / "results"
FIG.mkdir(exist_ok=True)
RES.mkdir(exist_ok=True)

TOP_N = 1000                   # number of products in the network (robustness: 2000)

## 1. Step 0 — Data preparation
**Goal:** load the six CSV files, join them, check that no rows are lost, split users 80/20, keep the top-N products.
Guide: chapter 6.

### 1.1 Load the raw files

In [ ]:
# TODO: load orders, order_products__train, order_products__prior (later, for SQ6),
#       products, aisles, departments.
# Hint: choose small dtypes (int32, int8, category) to save memory.

### 1.2 Join checks

In [ ]:
# TODO: merge baskets -> products -> aisles, departments.
# TODO: print the number of rows before and after each merge and the number of missing keys.

### 1.3 Split users 80 / 20
Split **by user**, not by row: no customer may appear on both sides. Everything is learned on the 80%;
SQ2, SQ4 and SQ5 are evaluated on the 20%.

In [ ]:
# TODO: GroupShuffleSplit(test_size=0.2, random_state=SEED), groups = user_id.
# TODO: check that the two user sets do not overlap.

### 1.4 Keep the top-N products

In [ ]:
# TODO: count baskets per product on TRAINING users only, keep the top TOP_N.
# TODO: what share of basket lines do they cover? (expected about 52.6% for N = 1000)

## 2. Step 1 — Exploratory data analysis
**Goal:** understand the data and justify our choices (top-N, minimum co-count k, lift threshold). Guide: chapter 7.

### 2.1 Basket size

In [ ]:
# TODO: histogram of products per basket; median and mean.

### 2.2 Department shares and reorder rate

In [ ]:
# TODO: share of items per department; reorder rate overall and by order number.

### 2.3 Product popularity and cumulative coverage of the top-N products

In [ ]:
# TODO: long-tail plot; cumulative share of basket lines covered by the top N products.

### 2.4 Purchase counts per product

In [ ]:
# TODO: how many products are bought fewer than 10 times?

### 2.5 Distribution of lift values

In [ ]:
# TODO: after the co-occurrence step, plot the distribution of lift (log scale).

**What we learned from the EDA (write in your own words):**

- ...

## 3. Step 2 — The co-purchase network
**Goal:** products are nodes; an edge joins two products bought together more often than chance.
lift(A,B) = P(A and B) / (P(A) · P(B)). Guide: chapters 8, 18, 19.

### 3.1 Basket × product matrix

In [ ]:
# TODO: sparse 0/1 matrix X (rows = training baskets, columns = top-N products).

### 3.2 Co-occurrence counts and lift

In [ ]:
# TODO: C = X.T @ X; compute lift from the counts. Which N do you divide by?

### 3.3 Threshold pilot

In [ ]:
# TODO: try several (k, lift) pairs; compare giant component, average degree, modularity. Choose and justify.

### 3.4 Build the graph and describe it

In [ ]:
# TODO: build G with networkx (edge weight = ?).
# TODO: nodes, edges, density, giant component, isolated products.
# TODO: compare clustering and average path length with a random (Erdos-Renyi) graph of the same size.

## 4. SQ1 — Communities vs the catalogue
*Which communities form via Louvain detection, and how closely do they match aisles and departments?* Guide: chapters 9, 20–22.

### 4.1 Louvain communities

In [ ]:
# TODO: nx.community.louvain_communities(G, resolution=..., seed=...). Aim for about 21 communities (= departments).

### 4.2 Products without edges

In [ ]:
# TODO: decide how isolated products get a community (every product needs a group).

### 4.3 NMI / ARI vs departments and aisles

In [ ]:
# TODO: sklearn normalized_mutual_info_score, adjusted_rand_score.

### 4.4 Permutation test

In [ ]:
# TODO: shuffle department labels 1,000 times; compare real NMI with the shuffled NMIs; p-value.

### 4.5 Describe and name the communities

In [ ]:
# TODO: top products and department mix per community; flag any community > 40% of products.

**SQ1 answer:**

- ...

## 5. SQ2 — Groups per basket
*Do held-out baskets span fewer groups under community grouping than under department grouping?* Guide: chapter 10.

In [ ]:
# TODO: for each held-out basket, count distinct groups under: communities, departments, aisles,
#       and a random grouping with the same group sizes (100 shuffles).

In [ ]:
# TODO: paired Wilcoxon signed-rank test (communities vs departments) + effect size.

**SQ2 answer:**

- ...

## 6. SQ3 — Central products
*Which products are most central (degree, closeness, betweenness)?* Guide: chapters 11, 23.

In [ ]:
# TODO: degree, closeness, betweenness (and participation coefficient) for every product.

In [ ]:
# TODO: Spearman correlation of each ranking with popularity; list anchors and bridges.

**SQ3 answer:**

- ...

## 7. SQ4 — The mini-market
*For a store with N products, which strategy (popularity / centrality / community) covers the most baskets?*
A basket is **served** if at least 80% of its items are stocked. Guide: chapters 12, 24.

In [ ]:
# TODO: write one function per strategy that returns a list of N products.

In [ ]:
# TODO: for N = 300, 500, 1000, share of held-out baskets served (80% and 100%); coverage curves.

In [ ]:
# TODO: bootstrap confidence interval for the difference between strategies.

**SQ4 answer:**

- ...

## 8. SQ5 [ML] — Predicting reorders
*Does community membership improve reorder prediction over department membership?* Guide: chapters 13, 25.

**Leakage rules:** never use "has this user bought this product before" (that is the target). Product statistics
come from training users only.

### 8.1 Features and target

In [ ]:
# TODO: target = reordered; shared features (cart position, basket size, order number, days since prior, popularity).

### 8.2 Models: baseline vs + department vs + community

In [ ]:
# TODO: logistic regression and gradient boosting for each feature set.

### 8.3 Evaluation

In [ ]:
# TODO: ROC-AUC (+ PR-AUC, F1); bootstrap confidence intervals over held-out USERS.

**SQ5 answer:**

- ...

## 9. SQ6 [ML] — Shopper segments
*Do K-means clusters of users by community-purchase vectors reveal interpretable shopper segments?*
Uses `order_products__prior.csv`. Guide: chapters 14, 26.

In [ ]:
# TODO: user vectors = share of purchases per community (and per department, for comparison).

In [ ]:
# TODO: K-means for k = 5..10; elbow, silhouette, stability across seeds (ARI).

In [ ]:
# TODO: profile and name each segment; do segments differ in held-out behaviour?

**SQ6 answer:**

- ...

## 10. Robustness checks
Repeat SQ1 and SQ2 with other thresholds, seeds, top 2,000 products. Guide: chapter 15.

In [ ]:
# TODO: loop over variants; table of number of communities, largest share, NMI, groups per basket.

## 11. Conclusions and data story
- Answer to the main question: ...
- What surprised us: ...
- Limits (US online data, no prices, undocumented order_dow, ...): ...
- Implications for catalogue designers and small stores: ...

Guide: chapter 16.